# 08 - Normalización de platos y construcción del catálogo inicial

## Objetivo

Este notebook inicia el módulo de normalización de platos de Hidden Gems.

Partimos de las menciones detectadas por el modelo `Dish NER v1`:

`dish_mentions_model_v1_full.jsonl`

El objetivo es pasar de menciones textuales sueltas a un catálogo inicial de platos:

```text
burger
burgers
cheeseburger
burger with bacon
→ burger / cheeseburger / bacon burger

taco
tacos
tuna tacos
→ tacos / tuna tacos

fries
french fries
home fries
→ fries
```

## Salidas esperadas

Este notebook generará artefactos como:

`dish_mentions_normalized_v1.jsonl`
`dish_surface_forms_v1.csv`
`dish_catalog_seed_v1.csv`
`dish_aliases_seed_v1.csv`
`dish_normalization_summary_v1.json`

Estos archivos serán la base del siguiente módulo:

09_dish_mention_sentiment.ipynb

## Enfoque

La normalización se hará de forma híbrida:

limpieza textual;
reglas lingüísticas simples;
agrupación singular/plural;
detección de variantes;
creación de familias de plato;
preparación para revisión manual futura.

Este primer normalizador no será perfecto, pero sí debe ser estable, trazable y útil para construir rankings iniciales.

In [5]:
# ============================================================
# 01. Imports y configuración base
# ============================================================

from pathlib import Path
import json
import random
import re
import os
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_colwidth", 300)

print("Entorno inicializado correctamente.")

Entorno inicializado correctamente.


In [6]:
# ============================================================
# 02. Detectar entorno y preparar carpetas
# ============================================================

IS_KAGGLE = Path("/kaggle/input").exists()
IS_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False

if IS_KAGGLE:
    ENV_NAME = "kaggle"
elif IS_COLAB:
    ENV_NAME = "colab"
else:
    ENV_NAME = "local"

print("Entorno detectado:", ENV_NAME)

if IS_KAGGLE:
    INPUT_DIR = Path("/kaggle/input")
    WORKING_DIR = Path("/kaggle/working")
    PROJECT_DIR = WORKING_DIR / "hidden_gems_ai"

elif IS_COLAB:
    from google.colab import drive

    try:
        !fusermount -u /content/drive 2>/dev/null || true
        !rm -rf /content/drive

        drive.mount("/content/drive", force_remount=True, timeout_ms=120000)

        PROJECT_DIR = Path("/content/drive/MyDrive/hidden_gems_ai")

    except Exception as e:
        print("No se ha podido montar Drive. Se usará almacenamiento temporal.")
        print("Error:", e)

        PROJECT_DIR = Path("/content/hidden_gems_ai")

else:
    PROJECT_DIR = Path.cwd()

OUTPUT_DIR = PROJECT_DIR / "outputs" / "dish_normalization"
MENTIONS_DIR = OUTPUT_DIR / "mentions"
CATALOG_DIR = OUTPUT_DIR / "catalog"
ALIASES_DIR = OUTPUT_DIR / "aliases"
ARTIFACTS_DIR = OUTPUT_DIR / "artifacts"
SAMPLES_DIR = OUTPUT_DIR / "samples"

for path in [
    PROJECT_DIR,
    OUTPUT_DIR,
    MENTIONS_DIR,
    CATALOG_DIR,
    ALIASES_DIR,
    ARTIFACTS_DIR,
    SAMPLES_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

Entorno detectado: kaggle
PROJECT_DIR: /kaggle/working/hidden_gems_ai
OUTPUT_DIR: /kaggle/working/hidden_gems_ai/outputs/dish_normalization


In [7]:
# ============================================================
# 03. Diagnóstico de archivos disponibles
# ============================================================

if IS_KAGGLE:
    print("Archivos disponibles en /kaggle/input:")

    for dirname, _, filenames in os.walk("/kaggle/input"):
        for filename in filenames:
            print(os.path.join(dirname, filename))
else:
    print("No estás en Kaggle. Se buscarán archivos en el proyecto local.")

Archivos disponibles en /kaggle/input:
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_labels.json
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_bio_high_precision_with_negatives.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_detection_manual_review_sample_v2_150.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/transformer_sentiment_metrics.json
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/weak_dish_candidates_v2.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_inference_summary_full.json
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_detection_exploration_summary_v2.json
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/yelp_translation_gold_seed_300_es_ready_v1.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_mentions_model_v1_full.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidde

In [8]:
# ============================================================
# 04. Localizar archivos necesarios
# ============================================================

MENTIONS_FILENAME = "dish_mentions_model_v1_full.jsonl"
INFERENCE_SUMMARY_FILENAME = "dish_ner_inference_summary_full.json"

def find_file(filename: str) -> Path:
    candidate_paths = []

    if IS_KAGGLE:
        candidate_paths.extend(list(Path("/kaggle/input").rglob(filename)))

    candidate_paths.extend(list(PROJECT_DIR.rglob(filename)))
    candidate_paths.extend(list(Path.cwd().rglob(filename)))

    candidate_paths = list(dict.fromkeys(candidate_paths))

    if not candidate_paths:
        raise FileNotFoundError(
            f"No se ha encontrado el archivo requerido: {filename}\n"
            "En Kaggle, asegúrate de haberlo añadido desde Add Data."
        )

    return candidate_paths[0]


MENTIONS_PATH = find_file(MENTIONS_FILENAME)

try:
    INFERENCE_SUMMARY_PATH = find_file(INFERENCE_SUMMARY_FILENAME)
except FileNotFoundError:
    INFERENCE_SUMMARY_PATH = None

print("MENTIONS_PATH:", MENTIONS_PATH)
print("INFERENCE_SUMMARY_PATH:", INFERENCE_SUMMARY_PATH)

MENTIONS_PATH: /kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_mentions_model_v1_full.jsonl
INFERENCE_SUMMARY_PATH: /kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_inference_summary_full.json


In [9]:
# ============================================================
# 05. Funciones auxiliares de carga y guardado
# ============================================================

def load_jsonl(path: Path) -> pd.DataFrame:
    records = []
    invalid_json_count = 0

    with path.open("r", encoding="utf-8") as f:
        for line in tqdm(f, desc=f"Leyendo {path.name}"):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                invalid_json_count += 1

    print(f"Archivo: {path.name}")
    print(f"Registros cargados: {len(records):,}")
    print(f"Líneas JSON inválidas: {invalid_json_count:,}")

    return pd.DataFrame(records)


def make_json_safe(value):
    if isinstance(value, (np.integer,)):
        return int(value)

    if isinstance(value, (np.floating,)):
        if np.isnan(value):
            return None
        return float(value)

    if isinstance(value, (np.bool_,)):
        return bool(value)

    if isinstance(value, float) and np.isnan(value):
        return None

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, set):
        return sorted(list(value))

    return value


def save_jsonl(df_to_save: pd.DataFrame, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        for record in df_to_save.to_dict(orient="records"):
            clean_record = {
                str(key): make_json_safe(value)
                for key, value in record.items()
            }

            f.write(json.dumps(clean_record, ensure_ascii=False) + "\n")

    print(f"Guardado: {path} ({len(df_to_save):,} registros)")

In [10]:
# ============================================================
# 06. Cargar menciones detectadas por Dish NER v1
# ============================================================

mentions_raw_df = load_jsonl(MENTIONS_PATH)

print("Shape:", mentions_raw_df.shape)

print("\nColumnas:")
print(mentions_raw_df.columns.tolist())

display(mentions_raw_df.head(5))

if INFERENCE_SUMMARY_PATH is not None:
    with INFERENCE_SUMMARY_PATH.open("r", encoding="utf-8") as f:
        inference_summary = json.load(f)

    print("\nResumen de inferencia:")
    print(json.dumps(inference_summary, indent=2, ensure_ascii=False)[:3000])

Leyendo dish_mentions_model_v1_full.jsonl: 0it [00:00, ?it/s]

Archivo: dish_mentions_model_v1_full.jsonl
Registros cargados: 95,061
Líneas JSON inválidas: 0
Shape: (95061, 31)

Columnas:
['mention_id', 'corpus_document_id', 'review_id', 'business_id', 'business_name', 'city', 'state', 'split', 'rating_value', 'sentiment_label', 'language', 'source_system_code', 'source_dataset', 'dish_mention_text', 'dish_mention_normalized', 'start_char', 'end_char', 'start_token', 'end_token', 'token_count', 'confidence_mean', 'confidence_min', 'confidence_max', 'tokens', 'review_text', 'was_review_truncated', 'model_name', 'model_checkpoint', 'inference_run_name', 'human_review_status', 'normalization_status']


,mention_id,corpus_document_id,review_id,business_id,business_name,city,state,split,rating_value,sentiment_label,language,source_system_code,source_dataset,dish_mention_text,dish_mention_normalized,start_char,end_char,start_token,end_token,token_count,confidence_mean,confidence_min,confidence_max,tokens,review_text,was_review_truncated,model_name,model_checkpoint,inference_run_name,human_review_status,normalization_status
0,dish_mention_c6a233f28ab0ec50a4bf,yelp_94c5a64cecd4448d105e5c8a,saUsX_uimxRlCVr67Z4Jig,YjUWPpI6HXG530lwP-fb2A,Kettle Restaurant,Tucson,AZ,train,3.0,neutral,en,yelp_open_dataset,yelp_open_dataset,tamale,tamale,88,94,18,19,1,0.997417,0.997417,0.997417,[tamale],"Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served all day long. Friendly, attentive staff. Good place for a casual relaxed meal with ...",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,pending
1,dish_mention_370e33f760028e4e87a3,yelp_69e10d25d69774ab39af6571,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,Zaika,Philadelphia,PA,train,5.0,positive,en,yelp_open_dataset,yelp_open_dataset,lamb curry,lamb curry,54,64,12,14,2,0.996373,0.993416,0.999331,"[lamb, curry]","Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,pending
2,dish_mention_2c51d464763786d9fef2,yelp_69e10d25d69774ab39af6571,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,Zaika,Philadelphia,PA,train,5.0,positive,en,yelp_open_dataset,yelp_open_dataset,korma,korma,69,74,15,16,1,0.997466,0.997466,0.997466,[korma],"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,pending
3,dish_mention_6cfa246893992204e58a,yelp_69e10d25d69774ab39af6571,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,Zaika,Philadelphia,PA,train,5.0,positive,en,yelp_open_dataset,yelp_open_dataset,naan,naan,103,107,22,23,1,0.998627,0.998627,0.998627,[naan],"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,pending
4,dish_mention_e468cc705c76b7c9e84f,yelp_95095495afc3a77ce68723a2,Sx8TMOWLNuJBWer-0pcmoA,e4Vwtrqf-wpJfwesgvdgxQ,Melt,New Orleans,LA,train,4.0,positive,en,yelp_open_dataset,yelp_open_dataset,sandwiches,sandwiches,185,195,38,39,1,0.999447,0.999447,0.999447,[sandwiches],"Cute interior and owner (?) gave us tour of upcoming patio/rooftop area which will be great on beautiful days like today. Cheese curds were very good and very filling. Really like that sandwiches come w salad, esp after eating too many curds! Had the onion, gruyere, tomato sandwich. Wasn't too m...",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,pending



Resumen de inferencia:
{
  "run_name": "full",
  "fast_dev_run": false,
  "total_reviews_processed": 79270,
  "total_mentions_detected": 95061,
  "reviews_with_mentions": 42471,
  "reviews_without_mentions": 36799,
  "truncated_reviews": 8279,
  "avg_mentions_per_review": 1.1992052478869686,
  "avg_mentions_per_review_with_mentions": 2.2382566928021475,
  "confidence": {
    "mean": 0.975224027584393,
    "median": 0.9990728795528412,
    "min": 0.35621654987335205,
    "max": 0.9997578263282776
  },
  "top_mentions": {
    "pizza": 8682,
    "burger": 5756,
    "fries": 4848,
    "sushi": 4461,
    "shrimp": 4398,
    "tacos": 2912,
    "steak": 2789,
    "ice cream": 2371,
    "burgers": 2334,
    "wings": 2213,
    "sandwiches": 2034,
    "oysters": 1691,
    "pancakes": 1536,
    "crab": 1530,
    "taco": 1475,
    "donuts": 1419,
    "pho": 1219,
    "salmon": 1134,
    "ribs": 1093,
    "burrito": 993,
    "fried chicken": 933,
    "tuna": 889,
    "french toast": 862,
    "donu

In [11]:
# ============================================================
# 07. Validación de columnas obligatorias
# ============================================================

required_cols = [
    "mention_id",
    "review_id",
    "business_id",
    "dish_mention_text",
    "dish_mention_normalized",
    "confidence_mean",
    "rating_value",
    "sentiment_label",
    "split",
    "review_text",
]

missing_cols = [
    col for col in required_cols
    if col not in mentions_raw_df.columns
]

if missing_cols:
    raise ValueError(f"Faltan columnas obligatorias: {missing_cols}")

print("Columnas obligatorias presentes.")

print("\nNulos principales:")
display(
    mentions_raw_df[required_cols]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nDuplicados mention_id:")
print(mentions_raw_df["mention_id"].duplicated().sum())

Columnas obligatorias presentes.

Nulos principales:


mention_id                 0
review_id                  0
business_id                0
dish_mention_text          0
dish_mention_normalized    0
confidence_mean            0
rating_value               0
sentiment_label            0
split                      0
review_text                0
dtype: int64


Duplicados mention_id:
0


In [12]:
# ============================================================
# 08. Preparar dataframe base de menciones
# ============================================================

mentions_df = mentions_raw_df.copy()

mentions_df["mention_id"] = mentions_df["mention_id"].astype(str)
mentions_df["review_id"] = mentions_df["review_id"].astype(str)
mentions_df["business_id"] = mentions_df["business_id"].astype(str)

mentions_df["dish_mention_text"] = (
    mentions_df["dish_mention_text"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

mentions_df["dish_mention_normalized"] = (
    mentions_df["dish_mention_normalized"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.lower()
    .str.strip()
)

mentions_df["confidence_mean"] = pd.to_numeric(
    mentions_df["confidence_mean"],
    errors="coerce"
)

mentions_df["rating_value"] = pd.to_numeric(
    mentions_df["rating_value"],
    errors="coerce"
)

mentions_df["sentiment_label"] = (
    mentions_df["sentiment_label"]
    .astype(str)
    .str.lower()
    .str.strip()
)

mentions_df["split"] = (
    mentions_df["split"]
    .astype(str)
    .str.lower()
    .str.strip()
)

mentions_df["mention_token_count"] = pd.to_numeric(
    mentions_df.get("token_count", np.nan),
    errors="coerce"
)

mentions_df["mention_char_length"] = mentions_df["dish_mention_normalized"].str.len()

# Limpieza básica
before_count = len(mentions_df)

mentions_df = mentions_df[
    mentions_df["dish_mention_normalized"].notna()
    & (mentions_df["dish_mention_normalized"].str.len() > 0)
    & mentions_df["confidence_mean"].notna()
].copy()

after_count = len(mentions_df)

print("Menciones antes:", before_count)
print("Menciones después:", after_count)
print("Descartadas:", before_count - after_count)

display(mentions_df.head(5))

Menciones antes: 95061
Menciones después: 95061
Descartadas: 0


,mention_id,corpus_document_id,review_id,business_id,business_name,city,state,split,rating_value,sentiment_label,language,source_system_code,source_dataset,dish_mention_text,dish_mention_normalized,start_char,end_char,start_token,end_token,token_count,confidence_mean,confidence_min,confidence_max,tokens,review_text,was_review_truncated,model_name,model_checkpoint,inference_run_name,human_review_status,normalization_status,mention_token_count,mention_char_length
0,dish_mention_c6a233f28ab0ec50a4bf,yelp_94c5a64cecd4448d105e5c8a,saUsX_uimxRlCVr67Z4Jig,YjUWPpI6HXG530lwP-fb2A,Kettle Restaurant,Tucson,AZ,train,3.0,neutral,en,yelp_open_dataset,yelp_open_dataset,tamale,tamale,88,94,18,19,1,0.997417,0.997417,0.997417,[tamale],"Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served all day long. Friendly, attentive staff. Good place for a casual relaxed meal with ...",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,pending,1,6
1,dish_mention_370e33f760028e4e87a3,yelp_69e10d25d69774ab39af6571,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,Zaika,Philadelphia,PA,train,5.0,positive,en,yelp_open_dataset,yelp_open_dataset,lamb curry,lamb curry,54,64,12,14,2,0.996373,0.993416,0.999331,"[lamb, curry]","Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,pending,2,10
2,dish_mention_2c51d464763786d9fef2,yelp_69e10d25d69774ab39af6571,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,Zaika,Philadelphia,PA,train,5.0,positive,en,yelp_open_dataset,yelp_open_dataset,korma,korma,69,74,15,16,1,0.997466,0.997466,0.997466,[korma],"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,pending,1,5
3,dish_mention_6cfa246893992204e58a,yelp_69e10d25d69774ab39af6571,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,Zaika,Philadelphia,PA,train,5.0,positive,en,yelp_open_dataset,yelp_open_dataset,naan,naan,103,107,22,23,1,0.998627,0.998627,0.998627,[naan],"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,pending,1,4
4,dish_mention_e468cc705c76b7c9e84f,yelp_95095495afc3a77ce68723a2,Sx8TMOWLNuJBWer-0pcmoA,e4Vwtrqf-wpJfwesgvdgxQ,Melt,New Orleans,LA,train,4.0,positive,en,yelp_open_dataset,yelp_open_dataset,sandwiches,sandwiches,185,195,38,39,1,0.999447,0.999447,0.999447,[sandwiches],"Cute interior and owner (?) gave us tour of upcoming patio/rooftop area which will be great on beautiful days like today. Cheese curds were very good and very filling. Really like that sandwiches come w salad, esp after eating too many curds! Had the onion, gruyere, tomato sandwich. Wasn't too m...",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,pending,1,10


In [13]:
# ============================================================
# 09. QA inicial de menciones
# ============================================================

qa_initial = {
    "total_mentions": int(len(mentions_df)),
    "unique_reviews": int(mentions_df["review_id"].nunique()),
    "unique_businesses": int(mentions_df["business_id"].nunique()),
    "unique_surface_forms": int(mentions_df["dish_mention_normalized"].nunique()),
    "confidence": {
        "min": float(mentions_df["confidence_mean"].min()),
        "mean": float(mentions_df["confidence_mean"].mean()),
        "median": float(mentions_df["confidence_mean"].median()),
        "max": float(mentions_df["confidence_mean"].max()),
    },
    "split_counts": {
        str(k): int(v)
        for k, v in mentions_df["split"].value_counts().to_dict().items()
    },
    "sentiment_counts": {
        str(k): int(v)
        for k, v in mentions_df["sentiment_label"].value_counts().to_dict().items()
    },
}

print(json.dumps(qa_initial, indent=2, ensure_ascii=False))

print("\nTop menciones normalizadas:")
display(
    mentions_df["dish_mention_normalized"]
    .value_counts()
    .head(50)
    .reset_index()
    .rename(columns={
        "index": "dish_mention_normalized",
        "dish_mention_normalized": "count"
    })
)

print("\nDistribución de confianza:")
display(mentions_df["confidence_mean"].describe())

print("\nDistribución longitud de mención:")
display(mentions_df["mention_char_length"].describe())

{
  "total_mentions": 95061,
  "unique_reviews": 42471,
  "unique_businesses": 4091,
  "unique_surface_forms": 10258,
  "confidence": {
    "min": 0.35621654987335205,
    "mean": 0.975224027584393,
    "median": 0.9990728795528412,
    "max": 0.9997578263282776
  },
  "split_counts": {
    "train": 76050,
    "validation": 9564,
    "test": 9447
  },
  "sentiment_counts": {
    "positive": 64727,
    "negative": 16411,
    "neutral": 13923
  }
}

Top menciones normalizadas:


,count,count
0,pizza,8682
1,burger,5756
2,fries,4848
3,sushi,4461
4,shrimp,4398
5,tacos,2912
6,steak,2789
7,ice cream,2371
8,burgers,2334
9,wings,2213



Distribución de confianza:


count    95061.000000
mean         0.975224
std          0.079815
min          0.356217
25%          0.997282
50%          0.999073
75%          0.999427
max          0.999758
Name: confidence_mean, dtype: float64


Distribución longitud de mención:


count    95061.000000
mean         8.236301
std          5.736856
min          1.000000
25%          5.000000
50%          6.000000
75%          9.000000
max         62.000000
Name: mention_char_length, dtype: float64

## 1. Normalización textual inicial

En esta primera fase se aplican reglas ligeras para limpiar las menciones.

La idea no es perder información, sino crear columnas adicionales:

- `mention_clean`: texto limpio de la mención;
- `mention_canonical_rule`: primera propuesta de forma canónica;
- `normalization_flags`: señales sobre posibles problemas;
- `is_noise_candidate`: posible mención no útil;
- `normalization_confidence_hint`: confianza aproximada de la normalización.

La mención original siempre se conserva.

In [14]:
# ============================================================
# 10. Reglas de limpieza textual
# ============================================================

LEADING_DETERMINERS = {
    "the", "a", "an", "some", "my", "our", "your", "their", "this", "that"
}

TRAILING_NOISE_WORDS = {
    "and", "or", "but", "with", "without", "for", "of", "to", "from"
}

# Términos que pueden ser comida, pero no son plato útil para ranking.
CONTEXT_ONLY_TERMS = {
    "food",
    "dish",
    "dishes",
    "meal",
    "menu",
    "appetizer",
    "appetizers",
    "starter",
    "starters",
    "entree",
    "entrees",
    "dessert",
    "desserts",
    "breakfast",
    "brunch",
}

# Términos no deseados si el modelo los detecta por error.
CLEAR_NOISE_TERMS = {
    "service",
    "staff",
    "server",
    "waiter",
    "waitress",
    "place",
    "restaurant",
    "atmosphere",
    "ambiance",
    "price",
    "prices",
    "drink",
    "drinks",
    "water",
    "table",
    "bathroom",
    "parking",
}

def basic_clean_mention(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = text.strip(" -_:;,.!?()[]{}\"'")

    words = text.split()

    while words and words[0] in LEADING_DETERMINERS:
        words = words[1:]

    while words and words[-1] in TRAILING_NOISE_WORDS:
        words = words[:-1]

    text = " ".join(words)
    text = text.strip(" -_:;,.!?()[]{}\"'")

    return text


def singularize_simple_token(token: str) -> str:
    """
    Singularización ligera para inglés.
    No pretende ser lingüísticamente perfecta.
    Solo reduce plurales frecuentes para agrupar variantes básicas.
    """

    irregular = {
        "fries": "fries",
        "tacos": "taco",
        "burgers": "burger",
        "sandwiches": "sandwich",
        "pizzas": "pizza",
        "wings": "wing",
        "ribs": "rib",
        "oysters": "oyster",
        "pancakes": "pancake",
        "donuts": "donut",
        "burritos": "burrito",
        "dumplings": "dumpling",
        "waffles": "waffle",
        "enchiladas": "enchilada",
        "quesadillas": "quesadilla",
        "tamales": "tamale",
        "noodles": "noodle",
        "eggs": "egg",
        "mussels": "mussel",
        "clams": "clam",
    }

    if token in irregular:
        return irregular[token]

    if len(token) > 4 and token.endswith("ies"):
        return token[:-3] + "y"

    if len(token) > 4 and token.endswith("es"):
        return token[:-2]

    if len(token) > 3 and token.endswith("s") and not token.endswith("ss"):
        return token[:-1]

    return token


def singularize_phrase_simple(text: str) -> str:
    words = text.split()

    if not words:
        return text

    # Singularizamos principalmente el núcleo final.
    words[-1] = singularize_simple_token(words[-1])

    return " ".join(words)


def detect_noise_flags(text: str) -> list[str]:
    flags = []

    if not text:
        flags.append("empty_after_cleaning")
        return flags

    if text in CONTEXT_ONLY_TERMS:
        flags.append("context_only_term")

    if text in CLEAR_NOISE_TERMS:
        flags.append("clear_noise_term")

    words = text.split()

    if len(words) > 7:
        flags.append("very_long_mention")

    if len(text) <= 2:
        flags.append("too_short")

    if any(word in CLEAR_NOISE_TERMS for word in words):
        flags.append("contains_noise_word")

    if re.search(r"\d", text):
        flags.append("contains_digit")

    return flags


def build_rule_canonical(text: str) -> str:
    clean = basic_clean_mention(text)
    canonical = singularize_phrase_simple(clean)
    canonical = basic_clean_mention(canonical)

    return canonical

In [15]:
# ============================================================
# 11. Aplicar normalización textual inicial
# ============================================================

mentions_df["mention_clean"] = mentions_df["dish_mention_normalized"].apply(
    basic_clean_mention
)

mentions_df["mention_canonical_rule"] = mentions_df["mention_clean"].apply(
    build_rule_canonical
)

mentions_df["normalization_flags"] = mentions_df["mention_canonical_rule"].apply(
    detect_noise_flags
)

mentions_df["is_noise_candidate"] = mentions_df["normalization_flags"].apply(
    lambda flags: len(flags) > 0
)

def compute_normalization_confidence_hint(row) -> float:
    confidence = float(row["confidence_mean"])

    if row["is_noise_candidate"]:
        confidence -= 0.25

    if row["mention_canonical_rule"] != row["mention_clean"]:
        confidence -= 0.02

    confidence = max(0.0, min(1.0, confidence))

    return round(confidence, 4)


mentions_df["normalization_confidence_hint"] = mentions_df.apply(
    compute_normalization_confidence_hint,
    axis=1
)

print("Menciones:", len(mentions_df))
print("Noise candidates:", int(mentions_df["is_noise_candidate"].sum()))

display(
    mentions_df[
        [
            "dish_mention_text",
            "dish_mention_normalized",
            "mention_clean",
            "mention_canonical_rule",
            "confidence_mean",
            "normalization_flags",
            "normalization_confidence_hint",
        ]
    ].head(20)
)

print("\nTop canonical rule:")
display(
    mentions_df["mention_canonical_rule"]
    .value_counts()
    .head(50)
    .reset_index()
    .rename(columns={
        "index": "mention_canonical_rule",
        "mention_canonical_rule": "count"
    })
)

Menciones: 95061
Noise candidates: 758


,dish_mention_text,dish_mention_normalized,mention_clean,mention_canonical_rule,confidence_mean,normalization_flags,normalization_confidence_hint
0,tamale,tamale,tamale,tamale,0.997417,[],0.9974
1,lamb curry,lamb curry,lamb curry,lamb curry,0.996373,[],0.9964
2,korma,korma,korma,korma,0.997466,[],0.9975
3,naan,naan,naan,naan,0.998627,[],0.9986
4,sandwiches,sandwiches,sandwiches,sandwich,0.999447,[],0.9794
5,wings,wings,wings,wing,0.999090,[],0.9791
6,sushi,sushi,sushi,sushi,0.999515,[],0.9995
7,gnocchi with marinara,gnocchi with marinara,gnocchi with marinara,gnocchi with marinara,0.850407,[],0.8504
8,baked eggplant appetizer,baked eggplant appetizer,baked eggplant appetizer,baked eggplant appetizer,0.773023,[],0.7730
9,Sonoran Dog,sonoran dog,sonoran dog,sonoran dog,0.986196,[],0.9862



Top canonical rule:


,count,count
0,pizza,8751
1,burger,8095
2,fries,4848
3,sushi,4470
4,taco,4436
5,shrimp,4421
6,steak,2794
7,ice cream,2411
8,wing,2215
9,donut,2175


In [16]:
# ============================================================
# 12. Crear tabla de surface forms
# ============================================================

surface_forms_df = (
    mentions_df
    .groupby(["dish_mention_normalized", "mention_clean", "mention_canonical_rule"], dropna=False)
    .agg(
        mention_count=("mention_id", "size"),
        review_count=("review_id", "nunique"),
        business_count=("business_id", "nunique"),
        avg_confidence=("confidence_mean", "mean"),
        min_confidence=("confidence_mean", "min"),
        max_confidence=("confidence_mean", "max"),
        avg_normalization_confidence=("normalization_confidence_hint", "mean"),
        avg_rating=("rating_value", "mean"),
        positive_mentions=("sentiment_label", lambda s: int((s == "positive").sum())),
        neutral_mentions=("sentiment_label", lambda s: int((s == "neutral").sum())),
        negative_mentions=("sentiment_label", lambda s: int((s == "negative").sum())),
        noise_candidate_count=("is_noise_candidate", "sum"),
    )
    .reset_index()
)

surface_forms_df["is_noise_candidate"] = (
    surface_forms_df["noise_candidate_count"] > 0
)

surface_forms_df = surface_forms_df.sort_values(
    ["mention_count", "avg_confidence"],
    ascending=[False, False]
).reset_index(drop=True)

print("Surface forms únicas:", len(surface_forms_df))

display(surface_forms_df.head(50))

Surface forms únicas: 10258


,dish_mention_normalized,mention_clean,mention_canonical_rule,mention_count,review_count,business_count,avg_confidence,min_confidence,max_confidence,avg_normalization_confidence,avg_rating,positive_mentions,neutral_mentions,negative_mentions,noise_candidate_count,is_noise_candidate
0,pizza,pizza,pizza,8682,4579,853,0.997651,0.510894,0.999738,0.997651,3.647662,5618,1140,1924,0,False
1,burger,burger,burger,5756,3458,718,0.992488,0.502620,0.999721,0.992488,3.660181,3612,1048,1096,0,False
2,fries,fries,fries,4848,3707,991,0.997342,0.508097,0.999758,0.997342,3.599216,2952,909,987,0,False
3,sushi,sushi,sushi,4461,2475,266,0.999148,0.881908,0.999708,0.999147,3.834566,3061,612,788,0,False
4,shrimp,shrimp,shrimp,4398,3259,865,0.995150,0.540861,0.999727,0.995149,3.903365,3153,549,696,0,False
5,tacos,tacos,taco,2912,2061,419,0.995019,0.504338,0.999729,0.975019,3.910027,2098,340,474,0,False
6,steak,steak,steak,2789,2101,829,0.994946,0.500823,0.999732,0.994948,3.558265,1658,441,690,0,False
7,ice cream,ice cream,ice cream,2371,1564,473,0.999370,0.978565,0.999612,0.999370,4.098271,1857,252,262,0,False
8,burgers,burgers,burger,2334,1871,506,0.999211,0.842032,0.999687,0.979211,3.746358,1548,413,373,0,False
9,wings,wings,wing,2213,1493,588,0.995358,0.526581,0.999702,0.975357,3.596927,1367,357,489,0,False


In [17]:
# ============================================================
# 13. Crear catálogo semilla por regla
# ============================================================

# Excluimos ruido claro del catálogo, pero lo mantendremos en menciones con flags.
catalog_source_df = surface_forms_df[
    ~surface_forms_df["is_noise_candidate"]
    & surface_forms_df["mention_canonical_rule"].notna()
    & (surface_forms_df["mention_canonical_rule"].str.len() > 0)
].copy()

catalog_seed_df = (
    catalog_source_df
    .groupby("mention_canonical_rule", dropna=False)
    .agg(
        total_mentions=("mention_count", "sum"),
        total_reviews=("review_count", "sum"),
        total_businesses=("business_count", "sum"),
        surface_form_count=("dish_mention_normalized", "nunique"),
        avg_confidence=("avg_confidence", "mean"),
        avg_rating=("avg_rating", "mean"),
        positive_mentions=("positive_mentions", "sum"),
        neutral_mentions=("neutral_mentions", "sum"),
        negative_mentions=("negative_mentions", "sum"),
    )
    .reset_index()
    .rename(columns={"mention_canonical_rule": "canonical_dish_name"})
)

catalog_seed_df["dish_id"] = [
    f"dish_seed_{idx:06d}"
    for idx in range(1, len(catalog_seed_df) + 1)
]

catalog_seed_df["language"] = "en"
catalog_seed_df["catalog_version"] = "dish_catalog_seed_v1"
catalog_seed_df["normalization_method"] = "rule_based_v1"
catalog_seed_df["manual_review_status"] = "pending"

# Prioridad de revisión manual
def compute_manual_review_priority(row):
    if row["total_mentions"] >= 500:
        return 3
    if row["total_mentions"] >= 100:
        return 2
    return 1

catalog_seed_df["manual_review_priority"] = catalog_seed_df.apply(
    compute_manual_review_priority,
    axis=1
)

catalog_seed_df = catalog_seed_df.sort_values(
    ["total_mentions", "surface_form_count"],
    ascending=[False, False]
).reset_index(drop=True)

# Reordenar dish_id tras ordenar
catalog_seed_df["dish_id"] = [
    f"dish_seed_{idx:06d}"
    for idx in range(1, len(catalog_seed_df) + 1)
]

print("Platos canónicos iniciales:", len(catalog_seed_df))

display(catalog_seed_df.head(50))

Platos canónicos iniciales: 9401


,canonical_dish_name,total_mentions,total_reviews,total_businesses,surface_form_count,avg_confidence,avg_rating,positive_mentions,neutral_mentions,negative_mentions,dish_id,language,catalog_version,normalization_method,manual_review_status,manual_review_priority
0,pizza,8751,4640,900,4,0.972458,4.005505,5661,1154,1936,dish_seed_000001,en,dish_catalog_seed_v1,rule_based_v1,pending,3
1,burger,8095,5334,1229,4,0.960858,3.726635,5164,1462,1469,dish_seed_000002,en,dish_catalog_seed_v1,rule_based_v1,pending,3
2,fries,4848,3707,991,1,0.997342,3.599216,2952,909,987,dish_seed_000003,en,dish_catalog_seed_v1,rule_based_v1,pending,3
3,sushi,4470,2484,275,3,0.999205,3.349617,3067,613,790,dish_seed_000004,en,dish_catalog_seed_v1,rule_based_v1,pending,3
4,taco,4436,3165,789,3,0.995807,3.759203,3003,571,862,dish_seed_000005,en,dish_catalog_seed_v1,rule_based_v1,pending,3
5,shrimp,4421,3281,881,4,0.819895,4.130603,3167,553,701,dish_seed_000006,en,dish_catalog_seed_v1,rule_based_v1,pending,3
6,steak,2794,2106,834,3,0.897557,2.852755,1660,441,693,dish_seed_000007,en,dish_catalog_seed_v1,rule_based_v1,pending,3
7,ice cream,2411,1604,499,2,0.998952,4.049135,1888,255,268,dish_seed_000008,en,dish_catalog_seed_v1,rule_based_v1,pending,3
8,wing,2215,1495,590,3,0.817018,4.198976,1369,357,489,dish_seed_000009,en,dish_catalog_seed_v1,rule_based_v1,pending,3
9,donut,2175,1258,225,2,0.997025,4.060152,1624,297,254,dish_seed_000010,en,dish_catalog_seed_v1,rule_based_v1,pending,3


In [18]:
# ============================================================
# 14. Crear aliases semilla
# ============================================================

dish_id_map = dict(
    zip(
        catalog_seed_df["canonical_dish_name"],
        catalog_seed_df["dish_id"]
    )
)

aliases_seed_df = surface_forms_df.copy()

aliases_seed_df = aliases_seed_df[
    aliases_seed_df["mention_canonical_rule"].isin(dish_id_map)
].copy()

aliases_seed_df["dish_id"] = aliases_seed_df["mention_canonical_rule"].map(dish_id_map)
aliases_seed_df["alias_text"] = aliases_seed_df["dish_mention_normalized"]
aliases_seed_df["canonical_dish_name"] = aliases_seed_df["mention_canonical_rule"]
aliases_seed_df["alias_language"] = "en"
aliases_seed_df["alias_source"] = "model_mentions_rule_based_v1"
aliases_seed_df["alias_type"] = np.where(
    aliases_seed_df["alias_text"] == aliases_seed_df["canonical_dish_name"],
    "canonical",
    "surface_variant"
)

aliases_seed_df["manual_review_status"] = "pending"

aliases_seed_df = aliases_seed_df[
    [
        "dish_id",
        "canonical_dish_name",
        "alias_text",
        "alias_type",
        "alias_language",
        "alias_source",
        "mention_count",
        "review_count",
        "business_count",
        "avg_confidence",
        "avg_normalization_confidence",
        "manual_review_status",
    ]
].copy()

aliases_seed_df = aliases_seed_df.sort_values(
    ["mention_count", "avg_confidence"],
    ascending=[False, False]
).reset_index(drop=True)

print("Aliases semilla:", len(aliases_seed_df))

display(aliases_seed_df.head(50))

Aliases semilla: 9692


,dish_id,canonical_dish_name,alias_text,alias_type,alias_language,alias_source,mention_count,review_count,business_count,avg_confidence,avg_normalization_confidence,manual_review_status
0,dish_seed_000001,pizza,pizza,canonical,en,model_mentions_rule_based_v1,8682,4579,853,0.997651,0.997651,pending
1,dish_seed_000002,burger,burger,canonical,en,model_mentions_rule_based_v1,5756,3458,718,0.992488,0.992488,pending
2,dish_seed_000003,fries,fries,canonical,en,model_mentions_rule_based_v1,4848,3707,991,0.997342,0.997342,pending
3,dish_seed_000004,sushi,sushi,canonical,en,model_mentions_rule_based_v1,4461,2475,266,0.999148,0.999147,pending
4,dish_seed_000006,shrimp,shrimp,canonical,en,model_mentions_rule_based_v1,4398,3259,865,0.995150,0.995149,pending
5,dish_seed_000005,taco,tacos,surface_variant,en,model_mentions_rule_based_v1,2912,2061,419,0.995019,0.975019,pending
6,dish_seed_000007,steak,steak,canonical,en,model_mentions_rule_based_v1,2789,2101,829,0.994946,0.994948,pending
7,dish_seed_000008,ice cream,ice cream,canonical,en,model_mentions_rule_based_v1,2371,1564,473,0.999370,0.999370,pending
8,dish_seed_000002,burger,burgers,surface_variant,en,model_mentions_rule_based_v1,2334,1871,506,0.999211,0.979211,pending
9,dish_seed_000009,wing,wings,surface_variant,en,model_mentions_rule_based_v1,2213,1493,588,0.995358,0.975357,pending


In [19]:
# ============================================================
# 15. Añadir dish_id y canonical_dish_name a cada mención
# ============================================================

mentions_normalized_df = mentions_df.copy()

mentions_normalized_df["canonical_dish_name"] = mentions_normalized_df["mention_canonical_rule"]
mentions_normalized_df["dish_id"] = mentions_normalized_df["canonical_dish_name"].map(dish_id_map)

mentions_normalized_df["normalization_status"] = np.where(
    mentions_normalized_df["dish_id"].notna(),
    "normalized_rule_based_v1",
    "excluded_or_pending_review"
)

mentions_normalized_df["normalization_method"] = np.where(
    mentions_normalized_df["dish_id"].notna(),
    "rule_based_v1",
    "none"
)

print("Menciones normalizadas:", int(mentions_normalized_df["dish_id"].notna().sum()))
print("Menciones pendientes/excluidas:", int(mentions_normalized_df["dish_id"].isna().sum()))

display(
    mentions_normalized_df[
        [
            "dish_mention_text",
            "dish_mention_normalized",
            "canonical_dish_name",
            "dish_id",
            "normalization_status",
            "confidence_mean",
            "normalization_flags",
            "review_id",
            "business_id",
        ]
    ].head(20)
)

Menciones normalizadas: 94303
Menciones pendientes/excluidas: 758


,dish_mention_text,dish_mention_normalized,canonical_dish_name,dish_id,normalization_status,confidence_mean,normalization_flags,review_id,business_id
0,tamale,tamale,tamale,dish_seed_000063,normalized_rule_based_v1,0.997417,[],saUsX_uimxRlCVr67Z4Jig,YjUWPpI6HXG530lwP-fb2A
1,lamb curry,lamb curry,lamb curry,dish_seed_000208,normalized_rule_based_v1,0.996373,[],AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA
2,korma,korma,korma,dish_seed_000078,normalized_rule_based_v1,0.997466,[],AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA
3,naan,naan,naan,dish_seed_000044,normalized_rule_based_v1,0.998627,[],AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA
4,sandwiches,sandwiches,sandwich,dish_seed_000012,normalized_rule_based_v1,0.999447,[],Sx8TMOWLNuJBWer-0pcmoA,e4Vwtrqf-wpJfwesgvdgxQ
5,wings,wings,wing,dish_seed_000009,normalized_rule_based_v1,0.999090,[],_ZeMknuYdlQcUqng_Im3yg,LHSTtnW3YHCeUkRDGyJOyw
6,sushi,sushi,sushi,dish_seed_000004,normalized_rule_based_v1,0.999515,[],pUycOfUwM8vqX7KjRRhUEA,gebiRewfieSdtt17PTW6Zg
7,gnocchi with marinara,gnocchi with marinara,gnocchi with marinara,dish_seed_004624,normalized_rule_based_v1,0.850407,[],8JFGBuHMoiNDyfcxuWNtrA,RZtGWDLCAtuipwaZ-UfjmQ
8,baked eggplant appetizer,baked eggplant appetizer,baked eggplant appetizer,dish_seed_001796,normalized_rule_based_v1,0.773023,[],8JFGBuHMoiNDyfcxuWNtrA,RZtGWDLCAtuipwaZ-UfjmQ
9,Sonoran Dog,sonoran dog,sonoran dog,dish_seed_000092,normalized_rule_based_v1,0.986196,[],UBp0zWyH60Hmw6Fsasei7w,otQS34_MymijPTdNBoBdCw


In [20]:
# ============================================================
# 16. Guardar outputs v1
# ============================================================

mentions_normalized_path = MENTIONS_DIR / "dish_mentions_normalized_v1.jsonl"
surface_forms_path = ARTIFACTS_DIR / "dish_surface_forms_v1.csv"
catalog_seed_path = CATALOG_DIR / "dish_catalog_seed_v1.csv"
aliases_seed_path = ALIASES_DIR / "dish_aliases_seed_v1.csv"
summary_path = ARTIFACTS_DIR / "dish_normalization_summary_v1.json"

save_jsonl(mentions_normalized_df, mentions_normalized_path)
surface_forms_df.to_csv(surface_forms_path, index=False)
catalog_seed_df.to_csv(catalog_seed_path, index=False)
aliases_seed_df.to_csv(aliases_seed_path, index=False)

normalization_summary = {
    "notebook": "08_dish_normalization_and_catalog_builder",
    "version": "v1_rule_based_seed",
    "source_mentions_path": str(MENTIONS_PATH),
    "input": qa_initial,
    "outputs": {
        "mentions_normalized": str(mentions_normalized_path),
        "surface_forms": str(surface_forms_path),
        "catalog_seed": str(catalog_seed_path),
        "aliases_seed": str(aliases_seed_path),
    },
    "normalization": {
        "total_mentions": int(len(mentions_normalized_df)),
        "normalized_mentions": int(mentions_normalized_df["dish_id"].notna().sum()),
        "pending_or_excluded_mentions": int(mentions_normalized_df["dish_id"].isna().sum()),
        "surface_forms": int(len(surface_forms_df)),
        "canonical_dishes_seed": int(len(catalog_seed_df)),
        "aliases_seed": int(len(aliases_seed_df)),
        "noise_candidate_mentions": int(mentions_normalized_df["is_noise_candidate"].sum()),
    },
    "top_canonical_dishes": (
        catalog_seed_df
        .head(50)
        .to_dict(orient="records")
    ),
}

with summary_path.open("w", encoding="utf-8") as f:
    json.dump(normalization_summary, f, indent=2, ensure_ascii=False)

print("Resumen guardado en:")
print(summary_path)

print(json.dumps(normalization_summary, indent=2, ensure_ascii=False)[:5000])

Guardado: /kaggle/working/hidden_gems_ai/outputs/dish_normalization/mentions/dish_mentions_normalized_v1.jsonl (95,061 registros)
Resumen guardado en:
/kaggle/working/hidden_gems_ai/outputs/dish_normalization/artifacts/dish_normalization_summary_v1.json
{
  "notebook": "08_dish_normalization_and_catalog_builder",
  "version": "v1_rule_based_seed",
  "source_mentions_path": "/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_mentions_model_v1_full.jsonl",
  "input": {
    "total_mentions": 95061,
    "unique_reviews": 42471,
    "unique_businesses": 4091,
    "unique_surface_forms": 10258,
    "confidence": {
      "min": 0.35621654987335205,
      "mean": 0.975224027584393,
      "median": 0.9990728795528412,
      "max": 0.9997578263282776
    },
    "split_counts": {
      "train": 76050,
      "validation": 9564,
      "test": 9447
    },
    "sentiment_counts": {
      "positive": 64727,
      "negative": 16411,
      "neutral": 13923
    }
  },
  "outputs": {
    

## 2. Refinamiento de normalización v2

La versión v1 ha generado una primera base trazable, pero algunas reglas de singularización han producido formas poco naturales o incorrectas.

Ejemplos detectados:

- `hummus` → `hummu`
- `crab cakes` → `crab cak`
- `wings` → `wing`
- `ribs` → `rib`
- `nachos` → `nacho`
- `mashed potatoes` → `mashed potato`

En este bloque se crea una versión v2 con:

1. correcciones manuales controladas;
2. aliases más consistentes;
3. formas canónicas más naturales;
4. detección de duplicados potenciales;
5. generación de nuevos artefactos v2.

In [21]:
# ============================================================
# 17. Reglas de corrección v2
# ============================================================

# Correcciones exactas sobre canonical_dish_name generado por v1.
CANONICAL_OVERRIDES_V2 = {
    # Errores claros de singularización
    "hummu": "hummus",
    "crab cak": "crab cakes",

    # Plurales naturales como platos
    "wing": "wings",
    "rib": "ribs",
    "nacho": "nachos",
    "mashed potato": "mashed potatoes",
    "spring roll": "spring rolls",
    "dumpling": "dumplings",
    "mussel": "mussels",
    "clam": "clams",
    "oyster": "oysters",

    # Variantes ortográficas
    "omelette": "omelet",
    "doughnut": "donut",
    "doughnuts": "donut",

    # Agrupaciones evidentes
    "french fries": "fries",
    "home fries": "fries",

    # Tacos suele funcionar mejor como familia plural en ranking
    "taco": "tacos",
}

# Correcciones sobre surface forms concretas.
SURFACE_FORM_OVERRIDES_V2 = {
    "hummus": "hummus",
    "crab cakes": "crab cakes",
    "crab cake": "crab cakes",
    "wings": "wings",
    "chicken wings": "chicken wings",
    "ribs": "ribs",
    "nachos": "nachos",
    "mashed potatoes": "mashed potatoes",
    "french fries": "fries",
    "home fries": "fries",
    "taco": "tacos",
    "tacos": "tacos",
    "omelette": "omelet",
    "omelet": "omelet",
}

# Términos que siguen siendo demasiado genéricos o poco útiles para ranking.
CONTEXT_OR_LOW_VALUE_DISH_TERMS_V2 = {
    "food",
    "dish",
    "dishes",
    "meal",
    "menu",
    "breakfast",
    "brunch",
    "appetizer",
    "appetizers",
    "starter",
    "starters",
    "entree",
    "entrees",
}

# Palabras que suelen indicar que una mención está demasiado extendida.
OVEREXTENSION_WORDS_V2 = {
    "with",
    "without",
    "and",
    "or",
    "plus",
    "side",
    "sides",
    "sauce",
    "sauces",
    "dressing",
    "topping",
    "toppings",
}


def normalize_canonical_v2(row) -> str:
    """
    Genera la forma canónica v2 combinando:
    - surface form original;
    - canonical rule v1;
    - overrides manuales controlados.
    """

    surface = str(row.get("dish_mention_normalized", "")).lower().strip()
    clean = str(row.get("mention_clean", "")).lower().strip()
    canonical_v1 = str(row.get("mention_canonical_rule", "")).lower().strip()

    # 1. Overrides por surface form exacta
    if surface in SURFACE_FORM_OVERRIDES_V2:
        return SURFACE_FORM_OVERRIDES_V2[surface]

    if clean in SURFACE_FORM_OVERRIDES_V2:
        return SURFACE_FORM_OVERRIDES_V2[clean]

    # 2. Overrides por canonical v1
    if canonical_v1 in CANONICAL_OVERRIDES_V2:
        return CANONICAL_OVERRIDES_V2[canonical_v1]

    # 3. Si no hay override, conservar v1
    return canonical_v1


def build_normalization_flags_v2(row) -> list[str]:
    flags = []

    canonical_v1 = str(row.get("mention_canonical_rule", "")).strip()
    canonical_v2 = str(row.get("canonical_dish_name_v2", "")).strip()
    surface = str(row.get("dish_mention_normalized", "")).strip()

    if not canonical_v2:
        flags.append("empty_canonical_v2")

    if canonical_v2 in CONTEXT_OR_LOW_VALUE_DISH_TERMS_V2:
        flags.append("context_or_low_value_term")

    if canonical_v1 != canonical_v2:
        flags.append("canonical_override_applied")

    words = canonical_v2.split()

    if len(words) > 6:
        flags.append("very_long_canonical")

    if any(word in OVEREXTENSION_WORDS_V2 for word in words):
        flags.append("possible_overextended_entity")

    if row.get("confidence_mean", 1.0) < 0.80:
        flags.append("low_model_confidence")

    # Casos típicos de stemming defectuoso
    if canonical_v2.endswith(" cak") or canonical_v2.endswith(" caks"):
        flags.append("bad_stemming_suspected")

    if canonical_v2.endswith("u") and surface.endswith("us"):
        flags.append("bad_stemming_suspected")

    return flags


def compute_v2_review_priority(row) -> int:
    """
    Prioridad de revisión manual:
    4 = revisar sí o sí
    3 = alta
    2 = media
    1 = baja
    """

    total_mentions = row.get("total_mentions", 0)
    surface_count = row.get("surface_form_count", 0)
    avg_confidence = row.get("avg_confidence", 1.0)
    canonical = str(row.get("canonical_dish_name", "")).lower()

    if total_mentions >= 1000:
        return 4

    if total_mentions >= 500:
        return 3

    if surface_count >= 5:
        return 3

    if avg_confidence < 0.85:
        return 3

    if any(word in canonical.split() for word in OVEREXTENSION_WORDS_V2):
        return 3

    if total_mentions >= 100:
        return 2

    return 1

In [23]:
# ============================================================
# 18. Aplicar normalización v2 a menciones
# ============================================================

mentions_v2_df = mentions_normalized_df.copy()

mentions_v2_df["canonical_dish_name_v1"] = mentions_v2_df["canonical_dish_name"]

mentions_v2_df["canonical_dish_name_v2"] = mentions_v2_df.apply(
    normalize_canonical_v2,
    axis=1
)

mentions_v2_df["normalization_flags_v2"] = mentions_v2_df.apply(
    build_normalization_flags_v2,
    axis=1
)

mentions_v2_df["is_noise_candidate_v2"] = mentions_v2_df["normalization_flags_v2"].apply(
    lambda flags: "context_or_low_value_term" in flags or "empty_canonical_v2" in flags
)

mentions_v2_df["normalization_status_v2"] = np.where(
    ~mentions_v2_df["is_noise_candidate_v2"]
    & mentions_v2_df["canonical_dish_name_v2"].notna()
    & (mentions_v2_df["canonical_dish_name_v2"].str.len() > 0),
    "normalized_rule_based_v2",
    "excluded_or_pending_review_v2"
)

mentions_v2_df["normalization_method_v2"] = np.where(
    mentions_v2_df["normalization_status_v2"] == "normalized_rule_based_v2",
    "rule_based_v2_overrides",
    "none"
)

print("Total menciones:", len(mentions_v2_df))
print("Normalizadas v2:", int((mentions_v2_df["normalization_status_v2"] == "normalized_rule_based_v2").sum()))
print("Pendientes/excluidas v2:", int((mentions_v2_df["normalization_status_v2"] != "normalized_rule_based_v2").sum()))

print("\nCambios de canonical v1 -> v2:")

changed_df = mentions_v2_df[
    mentions_v2_df["canonical_dish_name_v1"] != mentions_v2_df["canonical_dish_name_v2"]
].copy()

changed_display_df = changed_df[
    [
        "dish_mention_normalized",
        "canonical_dish_name_v1",
        "canonical_dish_name_v2",
        "confidence_mean",
        "normalization_flags_v2"
    ]
].copy()

# Convertimos la lista de flags a texto para poder usar drop_duplicates()
changed_display_df["normalization_flags_v2_str"] = changed_display_df["normalization_flags_v2"].apply(
    lambda flags: ", ".join(flags) if isinstance(flags, list) else str(flags)
)

changed_display_df = changed_display_df.drop(columns=["normalization_flags_v2"])

display(
    changed_display_df
    .drop_duplicates()
    .head(50)
)

Total menciones: 95061
Normalizadas v2: 94932
Pendientes/excluidas v2: 129

Cambios de canonical v1 -> v2:


,dish_mention_normalized,canonical_dish_name_v1,canonical_dish_name_v2,confidence_mean,normalization_flags_v2_str
5,wings,wing,wings,0.999090,canonical_override_applied
12,home fries,home fries,fries,0.999263,canonical_override_applied
40,mashed potatoes,mashed potato,mashed potatoes,0.999185,canonical_override_applied
47,taco,taco,tacos,0.999622,canonical_override_applied
58,nachos,nacho,nachos,0.998848,canonical_override_applied
61,nachos,nacho,nachos,0.998776,canonical_override_applied
65,wings,wing,wings,0.999503,canonical_override_applied
74,wings,wing,wings,0.996667,canonical_override_applied
98,oysters,oyster,oysters,0.999094,canonical_override_applied
99,oysters,oyster,oysters,0.998860,canonical_override_applied


In [24]:
# ============================================================
# 19. Recalcular surface forms v2
# ============================================================

surface_forms_v2_df = (
    mentions_v2_df
    .groupby(
        [
            "dish_mention_normalized",
            "mention_clean",
            "canonical_dish_name_v2"
        ],
        dropna=False
    )
    .agg(
        mention_count=("mention_id", "size"),
        review_count=("review_id", "nunique"),
        business_count=("business_id", "nunique"),
        avg_confidence=("confidence_mean", "mean"),
        min_confidence=("confidence_mean", "min"),
        max_confidence=("confidence_mean", "max"),
        avg_rating=("rating_value", "mean"),
        positive_mentions=("sentiment_label", lambda s: int((s == "positive").sum())),
        neutral_mentions=("sentiment_label", lambda s: int((s == "neutral").sum())),
        negative_mentions=("sentiment_label", lambda s: int((s == "negative").sum())),
        noise_candidate_count_v2=("is_noise_candidate_v2", "sum"),
    )
    .reset_index()
    .rename(columns={"canonical_dish_name_v2": "canonical_dish_name"})
)

surface_forms_v2_df["is_noise_candidate_v2"] = (
    surface_forms_v2_df["noise_candidate_count_v2"] > 0
)

surface_forms_v2_df = surface_forms_v2_df.sort_values(
    ["mention_count", "avg_confidence"],
    ascending=[False, False]
).reset_index(drop=True)

print("Surface forms v1:", len(surface_forms_df))
print("Surface forms v2:", len(surface_forms_v2_df))

display(surface_forms_v2_df.head(50))

Surface forms v1: 10258
Surface forms v2: 10258


,dish_mention_normalized,mention_clean,canonical_dish_name,mention_count,review_count,business_count,avg_confidence,min_confidence,max_confidence,avg_rating,positive_mentions,neutral_mentions,negative_mentions,noise_candidate_count_v2,is_noise_candidate_v2
0,pizza,pizza,pizza,8682,4579,853,0.997651,0.510894,0.999738,3.647662,5618,1140,1924,0,False
1,burger,burger,burger,5756,3458,718,0.992488,0.502620,0.999721,3.660181,3612,1048,1096,0,False
2,fries,fries,fries,4848,3707,991,0.997342,0.508097,0.999758,3.599216,2952,909,987,0,False
3,sushi,sushi,sushi,4461,2475,266,0.999148,0.881908,0.999708,3.834566,3061,612,788,0,False
4,shrimp,shrimp,shrimp,4398,3259,865,0.995150,0.540861,0.999727,3.903365,3153,549,696,0,False
5,tacos,tacos,tacos,2912,2061,419,0.995019,0.504338,0.999729,3.910027,2098,340,474,0,False
6,steak,steak,steak,2789,2101,829,0.994946,0.500823,0.999732,3.558265,1658,441,690,0,False
7,ice cream,ice cream,ice cream,2371,1564,473,0.999370,0.978565,0.999612,4.098271,1857,252,262,0,False
8,burgers,burgers,burger,2334,1871,506,0.999211,0.842032,0.999687,3.746358,1548,413,373,0,False
9,wings,wings,wings,2213,1493,588,0.995358,0.526581,0.999702,3.596927,1367,357,489,0,False


In [25]:
# ============================================================
# 20. Crear catálogo semilla v2
# ============================================================

catalog_source_v2_df = surface_forms_v2_df[
    ~surface_forms_v2_df["is_noise_candidate_v2"]
    & surface_forms_v2_df["canonical_dish_name"].notna()
    & (surface_forms_v2_df["canonical_dish_name"].str.len() > 0)
].copy()

catalog_v2_df = (
    catalog_source_v2_df
    .groupby("canonical_dish_name", dropna=False)
    .agg(
        total_mentions=("mention_count", "sum"),
        total_reviews=("review_count", "sum"),
        total_businesses=("business_count", "sum"),
        surface_form_count=("dish_mention_normalized", "nunique"),
        avg_confidence=("avg_confidence", "mean"),
        avg_rating=("avg_rating", "mean"),
        positive_mentions=("positive_mentions", "sum"),
        neutral_mentions=("neutral_mentions", "sum"),
        negative_mentions=("negative_mentions", "sum"),
    )
    .reset_index()
)

catalog_v2_df["language"] = "en"
catalog_v2_df["catalog_version"] = "dish_catalog_seed_v2"
catalog_v2_df["normalization_method"] = "rule_based_v2_overrides"
catalog_v2_df["manual_review_status"] = "pending"

catalog_v2_df["manual_review_priority"] = catalog_v2_df.apply(
    compute_v2_review_priority,
    axis=1
)

catalog_v2_df = catalog_v2_df.sort_values(
    ["total_mentions", "surface_form_count"],
    ascending=[False, False]
).reset_index(drop=True)

catalog_v2_df["dish_id"] = [
    f"dish_seed_v2_{idx:06d}"
    for idx in range(1, len(catalog_v2_df) + 1)
]

# Reordenar columnas
catalog_v2_df = catalog_v2_df[
    [
        "dish_id",
        "canonical_dish_name",
        "language",
        "catalog_version",
        "normalization_method",
        "total_mentions",
        "total_reviews",
        "total_businesses",
        "surface_form_count",
        "avg_confidence",
        "avg_rating",
        "positive_mentions",
        "neutral_mentions",
        "negative_mentions",
        "manual_review_status",
        "manual_review_priority",
    ]
].copy()

print("Platos canónicos v1:", len(catalog_seed_df))
print("Platos canónicos v2:", len(catalog_v2_df))

display(catalog_v2_df.head(60))

Platos canónicos v1: 9401
Platos canónicos v2: 9937


,dish_id,canonical_dish_name,language,catalog_version,normalization_method,total_mentions,total_reviews,total_businesses,surface_form_count,avg_confidence,avg_rating,positive_mentions,neutral_mentions,negative_mentions,manual_review_status,manual_review_priority
0,dish_seed_v2_000001,pizza,en,dish_catalog_seed_v2,rule_based_v2_overrides,8751,4640,900,4,0.972458,4.005505,5661,1154,1936,pending,4
1,dish_seed_v2_000002,burger,en,dish_catalog_seed_v2,rule_based_v2_overrides,8095,5334,1229,4,0.960858,3.726635,5164,1462,1469,pending,4
2,dish_seed_v2_000003,fries,en,dish_catalog_seed_v2,rule_based_v2_overrides,5375,4192,1314,3,0.998209,3.651662,3275,1010,1090,pending,4
3,dish_seed_v2_000004,sushi,en,dish_catalog_seed_v2,rule_based_v2_overrides,4470,2484,275,3,0.999205,3.349617,3067,613,790,pending,4
4,dish_seed_v2_000005,tacos,en,dish_catalog_seed_v2,rule_based_v2_overrides,4436,3165,789,3,0.995807,3.759203,3003,571,862,pending,4
5,dish_seed_v2_000006,shrimp,en,dish_catalog_seed_v2,rule_based_v2_overrides,4421,3281,881,4,0.819895,4.130603,3167,553,701,pending,4
6,dish_seed_v2_000007,steak,en,dish_catalog_seed_v2,rule_based_v2_overrides,2794,2106,834,3,0.897557,2.852755,1660,441,693,pending,4
7,dish_seed_v2_000008,ice cream,en,dish_catalog_seed_v2,rule_based_v2_overrides,2411,1604,499,2,0.998952,4.049135,1888,255,268,pending,4
8,dish_seed_v2_000009,wings,en,dish_catalog_seed_v2,rule_based_v2_overrides,2215,1495,590,3,0.817018,4.198976,1369,357,489,pending,4
9,dish_seed_v2_000010,donut,en,dish_catalog_seed_v2,rule_based_v2_overrides,2175,1258,225,2,0.997025,4.060152,1624,297,254,pending,4


In [26]:
# ============================================================
# 21. Crear aliases v2
# ============================================================

dish_id_map_v2 = dict(
    zip(
        catalog_v2_df["canonical_dish_name"],
        catalog_v2_df["dish_id"]
    )
)

aliases_v2_df = surface_forms_v2_df.copy()

aliases_v2_df = aliases_v2_df[
    aliases_v2_df["canonical_dish_name"].isin(dish_id_map_v2)
].copy()

aliases_v2_df["dish_id"] = aliases_v2_df["canonical_dish_name"].map(dish_id_map_v2)
aliases_v2_df["alias_text"] = aliases_v2_df["dish_mention_normalized"]
aliases_v2_df["alias_language"] = "en"
aliases_v2_df["alias_source"] = "model_mentions_rule_based_v2"

aliases_v2_df["alias_type"] = np.where(
    aliases_v2_df["alias_text"] == aliases_v2_df["canonical_dish_name"],
    "canonical",
    "surface_variant"
)

aliases_v2_df["manual_review_status"] = "pending"

aliases_v2_df = aliases_v2_df[
    [
        "dish_id",
        "canonical_dish_name",
        "alias_text",
        "alias_type",
        "alias_language",
        "alias_source",
        "mention_count",
        "review_count",
        "business_count",
        "avg_confidence",
        "manual_review_status",
    ]
].copy()

aliases_v2_df = aliases_v2_df.sort_values(
    ["mention_count", "avg_confidence"],
    ascending=[False, False]
).reset_index(drop=True)

print("Aliases v1:", len(aliases_seed_df))
print("Aliases v2:", len(aliases_v2_df))

display(aliases_v2_df.head(60))

Aliases v1: 9692
Aliases v2: 10235


,dish_id,canonical_dish_name,alias_text,alias_type,alias_language,alias_source,mention_count,review_count,business_count,avg_confidence,manual_review_status
0,dish_seed_v2_000001,pizza,pizza,canonical,en,model_mentions_rule_based_v2,8682,4579,853,0.997651,pending
1,dish_seed_v2_000002,burger,burger,canonical,en,model_mentions_rule_based_v2,5756,3458,718,0.992488,pending
2,dish_seed_v2_000003,fries,fries,canonical,en,model_mentions_rule_based_v2,4848,3707,991,0.997342,pending
3,dish_seed_v2_000004,sushi,sushi,canonical,en,model_mentions_rule_based_v2,4461,2475,266,0.999148,pending
4,dish_seed_v2_000006,shrimp,shrimp,canonical,en,model_mentions_rule_based_v2,4398,3259,865,0.995150,pending
5,dish_seed_v2_000005,tacos,tacos,canonical,en,model_mentions_rule_based_v2,2912,2061,419,0.995019,pending
6,dish_seed_v2_000007,steak,steak,canonical,en,model_mentions_rule_based_v2,2789,2101,829,0.994946,pending
7,dish_seed_v2_000008,ice cream,ice cream,canonical,en,model_mentions_rule_based_v2,2371,1564,473,0.999370,pending
8,dish_seed_v2_000002,burger,burgers,surface_variant,en,model_mentions_rule_based_v2,2334,1871,506,0.999211,pending
9,dish_seed_v2_000009,wings,wings,canonical,en,model_mentions_rule_based_v2,2213,1493,588,0.995358,pending


In [27]:
# ============================================================
# 22. Añadir dish_id v2 a cada mención
# ============================================================

mentions_v2_df["dish_id_v2"] = mentions_v2_df["canonical_dish_name_v2"].map(dish_id_map_v2)

mentions_v2_df["normalization_status_v2"] = np.where(
    mentions_v2_df["dish_id_v2"].notna(),
    "normalized_rule_based_v2",
    "excluded_or_pending_review_v2"
)

print("Menciones con dish_id_v2:", int(mentions_v2_df["dish_id_v2"].notna().sum()))
print("Menciones sin dish_id_v2:", int(mentions_v2_df["dish_id_v2"].isna().sum()))

display(
    mentions_v2_df[
        [
            "dish_mention_text",
            "dish_mention_normalized",
            "canonical_dish_name_v1",
            "canonical_dish_name_v2",
            "dish_id_v2",
            "normalization_status_v2",
            "confidence_mean",
            "normalization_flags_v2",
        ]
    ].head(30)
)

Menciones con dish_id_v2: 94932
Menciones sin dish_id_v2: 129


,dish_mention_text,dish_mention_normalized,canonical_dish_name_v1,canonical_dish_name_v2,dish_id_v2,normalization_status_v2,confidence_mean,normalization_flags_v2
0,tamale,tamale,tamale,tamale,dish_seed_v2_000060,normalized_rule_based_v2,0.997417,[]
1,lamb curry,lamb curry,lamb curry,lamb curry,dish_seed_v2_000205,normalized_rule_based_v2,0.996373,[]
2,korma,korma,korma,korma,dish_seed_v2_000074,normalized_rule_based_v2,0.997466,[]
3,naan,naan,naan,naan,dish_seed_v2_000044,normalized_rule_based_v2,0.998627,[]
4,sandwiches,sandwiches,sandwich,sandwich,dish_seed_v2_000012,normalized_rule_based_v2,0.999447,[]
5,wings,wings,wing,wings,dish_seed_v2_000009,normalized_rule_based_v2,0.999090,[canonical_override_applied]
6,sushi,sushi,sushi,sushi,dish_seed_v2_000004,normalized_rule_based_v2,0.999515,[]
7,gnocchi with marinara,gnocchi with marinara,gnocchi with marinara,gnocchi with marinara,dish_seed_v2_004917,normalized_rule_based_v2,0.850407,[possible_overextended_entity]
8,baked eggplant appetizer,baked eggplant appetizer,baked eggplant appetizer,baked eggplant appetizer,dish_seed_v2_001954,normalized_rule_based_v2,0.773023,[low_model_confidence]
9,Sonoran Dog,sonoran dog,sonoran dog,sonoran dog,dish_seed_v2_000088,normalized_rule_based_v2,0.986196,[]


In [28]:
# ============================================================
# 23. Detectar posibles duplicados para revisión
# ============================================================

!pip -q install rapidfuzz

from rapidfuzz import fuzz

# Para no hacer una comparación masiva 9k x 9k, trabajamos con candidatos relevantes.
duplicate_pool_df = catalog_v2_df[
    (catalog_v2_df["total_mentions"] >= 20)
    | (catalog_v2_df["manual_review_priority"] >= 2)
].copy()

names = duplicate_pool_df["canonical_dish_name"].tolist()

duplicate_candidate_rows = []

for i in tqdm(range(len(names)), desc="Buscando duplicados potenciales"):
    name_a = names[i]

    for j in range(i + 1, len(names)):
        name_b = names[j]

        # Evitar pares muy diferentes en longitud para reducir ruido
        if abs(len(name_a) - len(name_b)) > 15:
            continue

        ratio = fuzz.token_sort_ratio(name_a, name_b)
        partial = fuzz.partial_ratio(name_a, name_b)

        if ratio >= 92 or partial >= 96:
            row_a = duplicate_pool_df[duplicate_pool_df["canonical_dish_name"] == name_a].iloc[0]
            row_b = duplicate_pool_df[duplicate_pool_df["canonical_dish_name"] == name_b].iloc[0]

            duplicate_candidate_rows.append({
                "canonical_a": name_a,
                "canonical_b": name_b,
                "similarity_token_sort": ratio,
                "similarity_partial": partial,
                "mentions_a": int(row_a["total_mentions"]),
                "mentions_b": int(row_b["total_mentions"]),
                "review_priority_a": int(row_a["manual_review_priority"]),
                "review_priority_b": int(row_b["manual_review_priority"]),
                "suggested_action": "manual_review",
            })

duplicate_candidates_v2_df = pd.DataFrame(duplicate_candidate_rows)

if len(duplicate_candidates_v2_df) > 0:
    duplicate_candidates_v2_df = duplicate_candidates_v2_df.sort_values(
        ["similarity_token_sort", "similarity_partial", "mentions_a", "mentions_b"],
        ascending=[False, False, False, False]
    ).reset_index(drop=True)

print("Duplicados potenciales:", len(duplicate_candidates_v2_df))

display(duplicate_candidates_v2_df.head(100))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 41.0 MB/s eta 0:00:0000:0100:01


Buscando duplicados potenciales:   0%|          | 0/4299 [00:00<?, ?it/s]

Duplicados potenciales: 6984


,canonical_a,canonical_b,similarity_token_sort,similarity_partial,mentions_a,mentions_b,review_priority_a,review_priority_b,suggested_action
0,goat curry with sweet potato gnocchi,curry goat with sweet potato gnocchi,100.00000,92.537313,2,1,3,3,manual_review
1,caesar side salad,side caesar salad,100.00000,82.758621,1,1,3,3,manual_review
2,mango sticky rice,sticky mango rice,100.00000,78.571429,2,1,3,3,manual_review
3,shrimp tempura,tempura shrimp,100.00000,66.666667,62,30,1,1,manual_review
4,cheeseburger with bacon,cheeseburgers with bacon,97.87234,95.652174,1,1,3,3,manual_review
...,...,...,...,...,...,...,...,...,...
95,mac and cheese,mac and cheese bit,87.50000,100.000000,707,2,3,3,manual_review
96,mac and cheese,$ 5 mac and cheese,87.50000,100.000000,707,1,3,3,manual_review
97,mac and cheese,mac and cheese one,87.50000,100.000000,707,1,3,3,manual_review
98,ordered,i ordered,87.50000,100.000000,10,1,3,3,manual_review


In [29]:
# ============================================================
# 24. Crear muestras de revisión manual v2
# ============================================================

# Top platos del catálogo
catalog_review_top_df = catalog_v2_df.head(200).copy()

# Platos con baja confianza
catalog_review_low_conf_df = (
    catalog_v2_df
    .sort_values("avg_confidence", ascending=True)
    .head(200)
    .copy()
)

# Platos con muchas variantes surface
catalog_review_many_aliases_df = (
    catalog_v2_df
    .sort_values("surface_form_count", ascending=False)
    .head(200)
    .copy()
)

# Menciones marcadas con flags v2
mentions_review_flags_df = mentions_v2_df[
    mentions_v2_df["normalization_flags_v2"].apply(lambda flags: len(flags) > 0)
].copy()

mentions_review_flags_df = (
    mentions_review_flags_df
    .sort_values(["confidence_mean"], ascending=True)
    .head(300)
    .copy()
)

print("catalog_review_top_df:", len(catalog_review_top_df))
print("catalog_review_low_conf_df:", len(catalog_review_low_conf_df))
print("catalog_review_many_aliases_df:", len(catalog_review_many_aliases_df))
print("mentions_review_flags_df:", len(mentions_review_flags_df))

display(catalog_review_top_df.head(10))
display(catalog_review_low_conf_df.head(10))
display(catalog_review_many_aliases_df.head(10))
display(mentions_review_flags_df.head(10))

catalog_review_top_df: 200
catalog_review_low_conf_df: 200
catalog_review_many_aliases_df: 200
mentions_review_flags_df: 300


,dish_id,canonical_dish_name,language,catalog_version,normalization_method,total_mentions,total_reviews,total_businesses,surface_form_count,avg_confidence,avg_rating,positive_mentions,neutral_mentions,negative_mentions,manual_review_status,manual_review_priority
0,dish_seed_v2_000001,pizza,en,dish_catalog_seed_v2,rule_based_v2_overrides,8751,4640,900,4,0.972458,4.005505,5661,1154,1936,pending,4
1,dish_seed_v2_000002,burger,en,dish_catalog_seed_v2,rule_based_v2_overrides,8095,5334,1229,4,0.960858,3.726635,5164,1462,1469,pending,4
2,dish_seed_v2_000003,fries,en,dish_catalog_seed_v2,rule_based_v2_overrides,5375,4192,1314,3,0.998209,3.651662,3275,1010,1090,pending,4
3,dish_seed_v2_000004,sushi,en,dish_catalog_seed_v2,rule_based_v2_overrides,4470,2484,275,3,0.999205,3.349617,3067,613,790,pending,4
4,dish_seed_v2_000005,tacos,en,dish_catalog_seed_v2,rule_based_v2_overrides,4436,3165,789,3,0.995807,3.759203,3003,571,862,pending,4
5,dish_seed_v2_000006,shrimp,en,dish_catalog_seed_v2,rule_based_v2_overrides,4421,3281,881,4,0.819895,4.130603,3167,553,701,pending,4
6,dish_seed_v2_000007,steak,en,dish_catalog_seed_v2,rule_based_v2_overrides,2794,2106,834,3,0.897557,2.852755,1660,441,693,pending,4
7,dish_seed_v2_000008,ice cream,en,dish_catalog_seed_v2,rule_based_v2_overrides,2411,1604,499,2,0.998952,4.049135,1888,255,268,pending,4
8,dish_seed_v2_000009,wings,en,dish_catalog_seed_v2,rule_based_v2_overrides,2215,1495,590,3,0.817018,4.198976,1369,357,489,pending,4
9,dish_seed_v2_000010,donut,en,dish_catalog_seed_v2,rule_based_v2_overrides,2175,1258,225,2,0.997025,4.060152,1624,297,254,pending,4


,dish_id,canonical_dish_name,language,catalog_version,normalization_method,total_mentions,total_reviews,total_businesses,surface_form_count,avg_confidence,avg_rating,positive_mentions,neutral_mentions,negative_mentions,manual_review_status,manual_review_priority
7003,dish_seed_v2_007004,pinwheel,en,dish_catalog_seed_v2,rule_based_v2_overrides,1,1,1,1,0.434031,4.0,1,0,0,pending,3
8670,dish_seed_v2_008671,st,en,dish_catalog_seed_v2,rule_based_v2_overrides,1,1,1,1,0.459022,2.0,0,0,1,pending,3
1939,dish_seed_v2_001940,baguette,en,dish_catalog_seed_v2,rule_based_v2_overrides,1,1,1,1,0.462592,4.0,1,0,0,pending,3
4884,dish_seed_v2_004885,gilled,en,dish_catalog_seed_v2,rule_based_v2_overrides,1,1,1,1,0.463758,4.0,1,0,0,pending,3
1523,dish_seed_v2_001524,tso,en,dish_catalog_seed_v2,rule_based_v2_overrides,2,2,2,1,0.467482,2.5,1,0,1,pending,3
1644,dish_seed_v2_001645,2-salad,en,dish_catalog_seed_v2,rule_based_v2_overrides,1,1,1,1,0.467583,3.0,0,1,0,pending,3
8302,dish_seed_v2_008303,slaw,en,dish_catalog_seed_v2,rule_based_v2_overrides,1,1,1,1,0.489394,5.0,1,0,0,pending,3
6822,dish_seed_v2_006823,patrol,en,dish_catalog_seed_v2,rule_based_v2_overrides,1,1,1,1,0.490808,4.0,1,0,0,pending,3
1741,dish_seed_v2_001742,all,en,dish_catalog_seed_v2,rule_based_v2_overrides,1,1,1,1,0.494391,5.0,1,0,0,pending,3
6285,dish_seed_v2_006286,mother,en,dish_catalog_seed_v2,rule_based_v2_overrides,1,1,1,1,0.494748,5.0,1,0,0,pending,3


,dish_id,canonical_dish_name,language,catalog_version,normalization_method,total_mentions,total_reviews,total_businesses,surface_form_count,avg_confidence,avg_rating,positive_mentions,neutral_mentions,negative_mentions,manual_review_status,manual_review_priority
23,dish_seed_v2_000024,nachos,en,dish_catalog_seed_v2,rule_based_v2_overrides,764,611,311,5,0.992353,3.347742,493,117,154,pending,3
0,dish_seed_v2_000001,pizza,en,dish_catalog_seed_v2,rule_based_v2_overrides,8751,4640,900,4,0.972458,4.005505,5661,1154,1936,pending,4
61,dish_seed_v2_000062,salad,en,dish_catalog_seed_v2,rule_based_v2_overrides,176,173,154,4,0.741223,3.260174,105,30,41,pending,3
34,dish_seed_v2_000035,dumplings,en,dish_catalog_seed_v2,rule_based_v2_overrides,498,407,204,4,0.940333,3.945602,366,64,68,pending,2
22,dish_seed_v2_000023,omelet,en,dish_catalog_seed_v2,rule_based_v2_overrides,779,674,359,4,0.955139,3.877472,534,122,123,pending,3
11,dish_seed_v2_000012,sandwich,en,dish_catalog_seed_v2,rule_based_v2_overrides,2127,1870,882,4,0.752680,3.654794,1595,269,263,pending,4
5,dish_seed_v2_000006,shrimp,en,dish_catalog_seed_v2,rule_based_v2_overrides,4421,3281,881,4,0.819895,4.130603,3167,553,701,pending,4
42,dish_seed_v2_000043,eggplant,en,dish_catalog_seed_v2,rule_based_v2_overrides,381,329,199,4,0.929995,4.619695,288,55,38,pending,2
1,dish_seed_v2_000002,burger,en,dish_catalog_seed_v2,rule_based_v2_overrides,8095,5334,1229,4,0.960858,3.726635,5164,1462,1469,pending,4
122,dish_seed_v2_000123,caesar,en,dish_catalog_seed_v2,rule_based_v2_overrides,21,20,20,3,0.595430,4.017544,11,3,7,pending,3


,mention_id,corpus_document_id,review_id,business_id,business_name,city,state,split,rating_value,sentiment_label,language,source_system_code,source_dataset,dish_mention_text,dish_mention_normalized,start_char,end_char,start_token,end_token,token_count,confidence_mean,confidence_min,confidence_max,tokens,review_text,was_review_truncated,model_name,model_checkpoint,inference_run_name,human_review_status,normalization_status,mention_token_count,mention_char_length,mention_clean,mention_canonical_rule,normalization_flags,is_noise_candidate,normalization_confidence_hint,canonical_dish_name,dish_id,normalization_method,canonical_dish_name_v1,canonical_dish_name_v2,normalization_flags_v2,is_noise_candidate_v2,normalization_status_v2,normalization_method_v2,dish_id_v2
19337,dish_mention_54624608d71a1e8d2832,yelp_39a61463c6f167fdd244d064,_Pj3AEJ5DU2ScrnJwmwkFQ,6CsLAjWXFlLpErBZEmGQ5g,Sky Cafe,Philadelphia,PA,train,4.0,positive,en,yelp_open_dataset,yelp_open_dataset,tempeh,tempeh,479,485,100,101,1,0.356217,0.356217,0.356217,[tempeh],"Great little Indonesian restaurant near Cacias bakery on corner of Ritner & 16th. There is a sign inside warning not to park in front of Cacias. Mostly Indonesian clientele. Food was really good and came out fast. Menu is divided into noodle(w/ broth or stir-fried), rice dishes, a few starters a...",True,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,normalized_rule_based_v1,1,6,tempeh,tempeh,[],False,0.3562,tempeh,dish_seed_000238,rule_based_v1,tempeh,tempeh,[low_model_confidence],False,normalized_rule_based_v2,rule_based_v2_overrides,dish_seed_v2_000235
71605,dish_mention_8f89d5b7fa9c0a6ef9b0,yelp_7619577b390c28556a51de64,C30mh2VJ5VmCGdtqvYILVw,1LVMlrAQeqLgiwFu06kFfw,The Workshop Eatery,Edmonton,AB,validation,3.0,neutral,en,yelp_open_dataset,yelp_open_dataset,loin,loin,718,722,144,145,1,0.366022,0.366022,0.366022,[loin],My wife and I decided to give Workshop Eatery a try after a friend recommended it. We arrived 5:15pm on a Thursday and were promptly seated. The server asked for our drink order and delivered it 10 minutes later at which time she took our order. We started with the Arancini which was very good m...,False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,normalized_rule_based_v1,1,4,loin,loin,[],False,0.3660,loin,dish_seed_000762,rule_based_v1,loin,loin,[low_model_confidence],False,normalized_rule_based_v2,rule_based_v2_overrides,dish_seed_v2_000775
2599,dish_mention_9827050706b82291ce95,yelp_6d425329644c5061f553c795,wKVScyIiwc3cFwyNqLedAw,Gw7UW0E2BguzL9suQnwDeg,SPOT Gourmet Burgers,Philadelphia,PA,train,4.0,positive,en,yelp_open_dataset,yelp_open_dataset,bun,bun,967,970,202,203,1,0.397663,0.397663,0.397663,[bun],"Spot has been on my burger list to try for sometime. Spot burger consistently shows up on numerous best of Philly burger lists. When they opened up shop in Brewery town, I quickly bookmarked Spot and waited for a burger craving to hit me to give it a try. Spot is on West Girard Ave and I was abl...",True,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,normalized_rule_based_v1,1,3,bun,bun,[],False,0.3977,bun,dish_seed_000522,rule_based_v1,bun,bun,[low_model_confidence],False,normalized_rule_based_v2,rule_based_v2_overrides,dish_seed_v2_000529
32966,dish_mention_f83f8e98e3b0e17e5d99,yelp_2f79c62ace9b56eac5db0609,T-qChKcyxWjU5qZEEd4kxA,mCo2uVTTGYrEhRrkQW-CMw,Empress Garden,Philadelphia,PA,train,5.0,positive,en,yelp_open_dataset,yelp_open_dataset,noodle,noodle,210,216,45,46,1,0.398607,0.398607,0.398607,[noodle],"I've been going here since I was a kid. The same waitress that served me twenty years ago is still there. Fast, courteous, even too polite at times. Went today for lunch to get my favorite dish- Taiwanese beef noodle soup. This 

In [30]:
# ============================================================
# 25. Guardar outputs v2
# ============================================================

mentions_normalized_v2_path = MENTIONS_DIR / "dish_mentions_normalized_v2.jsonl"
surface_forms_v2_path = ARTIFACTS_DIR / "dish_surface_forms_v2.csv"
catalog_v2_path = CATALOG_DIR / "dish_catalog_seed_v2.csv"
aliases_v2_path = ALIASES_DIR / "dish_aliases_seed_v2.csv"
duplicate_candidates_v2_path = ARTIFACTS_DIR / "dish_normalization_duplicate_candidates_v2.csv"

catalog_review_top_path = SAMPLES_DIR / "dish_catalog_review_top_v2.csv"
catalog_review_low_conf_path = SAMPLES_DIR / "dish_catalog_review_low_confidence_v2.csv"
catalog_review_many_aliases_path = SAMPLES_DIR / "dish_catalog_review_many_aliases_v2.csv"
mentions_review_flags_path = SAMPLES_DIR / "dish_mentions_review_flags_v2.jsonl"

summary_v2_path = ARTIFACTS_DIR / "dish_normalization_summary_v2.json"

save_jsonl(mentions_v2_df, mentions_normalized_v2_path)
surface_forms_v2_df.to_csv(surface_forms_v2_path, index=False)
catalog_v2_df.to_csv(catalog_v2_path, index=False)
aliases_v2_df.to_csv(aliases_v2_path, index=False)

if len(duplicate_candidates_v2_df) > 0:
    duplicate_candidates_v2_df.to_csv(duplicate_candidates_v2_path, index=False)
else:
    pd.DataFrame(columns=[
        "canonical_a",
        "canonical_b",
        "similarity_token_sort",
        "similarity_partial",
        "mentions_a",
        "mentions_b",
        "review_priority_a",
        "review_priority_b",
        "suggested_action",
    ]).to_csv(duplicate_candidates_v2_path, index=False)

catalog_review_top_df.to_csv(catalog_review_top_path, index=False)
catalog_review_low_conf_df.to_csv(catalog_review_low_conf_path, index=False)
catalog_review_many_aliases_df.to_csv(catalog_review_many_aliases_path, index=False)
save_jsonl(mentions_review_flags_df, mentions_review_flags_path)

normalization_summary_v2 = {
    "notebook": "08_dish_normalization_and_catalog_builder",
    "version": "v2_rule_based_overrides",
    "source_mentions_path": str(MENTIONS_PATH),
    "input": qa_initial,
    "v1": {
        "normalized_mentions": int(mentions_normalized_df["dish_id"].notna().sum()),
        "pending_or_excluded_mentions": int(mentions_normalized_df["dish_id"].isna().sum()),
        "surface_forms": int(len(surface_forms_df)),
        "canonical_dishes_seed": int(len(catalog_seed_df)),
        "aliases_seed": int(len(aliases_seed_df)),
    },
    "v2": {
        "normalized_mentions": int(mentions_v2_df["dish_id_v2"].notna().sum()),
        "pending_or_excluded_mentions": int(mentions_v2_df["dish_id_v2"].isna().sum()),
        "surface_forms": int(len(surface_forms_v2_df)),
        "canonical_dishes_seed": int(len(catalog_v2_df)),
        "aliases_seed": int(len(aliases_v2_df)),
        "duplicate_candidates": int(len(duplicate_candidates_v2_df)),
        "mentions_with_flags": int(len(mentions_review_flags_df)),
    },
    "outputs": {
        "mentions_normalized_v2": str(mentions_normalized_v2_path),
        "surface_forms_v2": str(surface_forms_v2_path),
        "catalog_seed_v2": str(catalog_v2_path),
        "aliases_seed_v2": str(aliases_v2_path),
        "duplicate_candidates_v2": str(duplicate_candidates_v2_path),
        "catalog_review_top_v2": str(catalog_review_top_path),
        "catalog_review_low_confidence_v2": str(catalog_review_low_conf_path),
        "catalog_review_many_aliases_v2": str(catalog_review_many_aliases_path),
        "mentions_review_flags_v2": str(mentions_review_flags_path),
    },
    "top_canonical_dishes_v2": (
        catalog_v2_df
        .head(50)
        .to_dict(orient="records")
    ),
}

with summary_v2_path.open("w", encoding="utf-8") as f:
    json.dump(normalization_summary_v2, f, indent=2, ensure_ascii=False)

print("Resumen v2 guardado en:")
print(summary_v2_path)

print(json.dumps(normalization_summary_v2, indent=2, ensure_ascii=False)[:5000])

Guardado: /kaggle/working/hidden_gems_ai/outputs/dish_normalization/mentions/dish_mentions_normalized_v2.jsonl (95,061 registros)
Guardado: /kaggle/working/hidden_gems_ai/outputs/dish_normalization/samples/dish_mentions_review_flags_v2.jsonl (300 registros)
Resumen v2 guardado en:
/kaggle/working/hidden_gems_ai/outputs/dish_normalization/artifacts/dish_normalization_summary_v2.json
{
  "notebook": "08_dish_normalization_and_catalog_builder",
  "version": "v2_rule_based_overrides",
  "source_mentions_path": "/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_mentions_model_v1_full.jsonl",
  "input": {
    "total_mentions": 95061,
    "unique_reviews": 42471,
    "unique_businesses": 4091,
    "unique_surface_forms": 10258,
    "confidence": {
      "min": 0.35621654987335205,
      "mean": 0.975224027584393,
      "median": 0.9990728795528412,
      "max": 0.9997578263282776
    },
    "split_counts": {
      "train": 76050,
      "validation": 9564,
      "test": 9447


## Cierre provisional del bloque 2

En este bloque se ha generado una versión v2 del normalizador de platos.

La v2 corrige errores claros de la v1, especialmente derivados de una singularización demasiado agresiva:

- `hummu` → `hummus`
- `crab cak` → `crab cakes`
- `wing` → `wings`
- `rib` → `ribs`
- `nacho` → `nachos`
- `mashed potato` → `mashed potatoes`
- `french fries` → `fries`

También se han generado candidatos de duplicados y muestras de revisión manual.

El archivo principal para continuar el flujo será:

`dish_mentions_normalized_v2.jsonl`

Este archivo servirá como entrada para el siguiente módulo:

`09_dish_mention_sentiment.ipynb`